## load data

In [0]:
import pyspark.sql.functions as f
df=spark.read.table("dwh_project.bronze.prd_info")


## all the column transformations 

In [0]:
df= ( df.withColumn(
        "cat_id",
                f.regexp_replace(f.substring(f.col("prd_key"),1,5),'-','_')
    )
     .withColumn(
         "prd_key",
            f.substring(f.col("prd_key"),7,f.length("prd_key"))
         )
     .withColumn(
         "prd_cost",
            f.when(f.col("prd_cost").isNull(),0)
            .otherwise(f.col("prd_cost"))
         )
     .withColumn(
         "prd_line",
                f.when(f.upper(f.trim(f.col("prd_line")))=='R',"Road")
                .when(f.upper(f.trim(f.col("prd_line")))=='M',"Mountain")
                .when(f.upper(f.trim(f.col("prd_line")))=='S',"other sales")
                .when(f.upper(f.trim(f.col("prd_line")))=='T',"Touring")
                .otherwise("n/a")
        )
)


In [0]:
from pyspark.sql.window import Window
logic =(
    Window.partitionBy("prd_key")
    .orderBy(f.col("prd_start_dt").asc())
)
df=(
    df.withColumn("prd_end_dt", f.lead("prd_start_dt",1).over(logic))
)


In [0]:
df=df.select("prd_id","cat_id","prd_key","prd_nm","prd_cost","prd_line","prd_start_dt","prd_end_dt")

In [0]:
df.write.mode("overwrite").saveAsTable("dwh_project.silver.prd_info")